# Unfrozen Encoder Probe — Google Colab

Runs the ordinal-vs-InfoNCE **unfrozen encoder** probe on the v7 dataset.

**Before you start:** Runtime → Change runtime type → GPU (A100 recommended for seq-len 512; T4/L4/A100-40GB use seq-len 256).

You will need:
- A Hugging Face **read** token (data lives in the private dataset `olukotunjosh/cdcl-v7-data`).
- If the GitHub repo is private, a GitHub token with repo read access.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Clone the repo (branch: ordinal-fix-and-unfrozen-probe)

In [ ]:
import os

BRANCH = 'ordinal-fix-and-unfrozen-probe'
REPO   = 'github.com/tysjosh/job-matching-contrastive-learning.git'
REPO_DIR = '/content/job-matching-contrastive-learning'

# If the repo is PRIVATE, paste a GitHub token when prompted (leave blank if public).
from getpass import getpass
gh_token = getpass('GitHub token (blank if public): ').strip()

clone_url = f'https://{gh_token + "@" if gh_token else ""}{REPO}'

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
!git log --oneline -1

## 3. Install dependencies

In [ ]:
%pip install -q sentence-transformers networkx rapidfuzz huggingface_hub scikit-learn scipy pandas

## 4. Authenticate with Hugging Face and download the data

In [ ]:
from huggingface_hub import login, snapshot_download
from getpass import getpass

hf_token = getpass('Hugging Face READ token: ').strip()
login(hf_token)

# Download data splits into preprocess/data_splits_v7/ (paths preserved).
snapshot_download(
    repo_id='olukotunjosh/cdcl-v7-data', repo_type='dataset',
    allow_patterns=['data_splits_v7/*'], local_dir='preprocess',
)
print('Data files:')
!ls -lh preprocess/data_splits_v7

## 5. Generate the unfrozen probe configs

- `MAX_SEQ = 256` for 40GB GPUs (T4/L4/A100-40GB).
- `MAX_SEQ = 512` on an **A100 80GB** for parity with the frozen runs.
- `BATCH = 64` matches the frozen EO/E4 runs.

In [ ]:
BATCH   = 64
MAX_SEQ = 256   # set to 512 if on an A100 80GB
EPOCHS  = 15
LR      = 2e-5
SEEDS   = '13 21 42 87 123'

!python3 scripts/generate_unfrozen_probe.py \
    --epochs {EPOCHS} --lr {LR} --batch-size {BATCH} --max-seq-length {MAX_SEQ} \
    --seeds {SEEDS} --execute-list run_unfrozen_probe_gpu.sh

import json
cfg = json.load(open('results/research_runs/UF-Ordinal__cnamuangtoun__s13/training_config.json'))
print('batch_size:', cfg['batch_size'], '| unfrozen_max_seq_length:', cfg['unfrozen_max_seq_length'],
      '| freeze_text_encoder:', cfg['freeze_text_encoder'])

## 6. Run the probe (10 runs: UF-Ordinal + UF-InfoNCE × 5 seeds)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # reduce fragmentation
!bash run_unfrozen_probe_gpu.sh

## 7. Aggregate results (frozen-vs-unfrozen table)

In [ ]:
!python3 scripts/aggregate_eo_results.py

## 8. (Optional) Save results to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, glob, os
dest = '/content/drive/MyDrive/cdcl_uf_results'
os.makedirs(dest, exist_ok=True)

# Copy the small eval JSONs + the aggregated summary (skip large .pt checkpoints).
for f in glob.glob('results/research_runs/UF-*/phase1_evaluation/ordinal_evaluation_results.json'):
    run = f.split('/')[2]
    os.makedirs(f'{dest}/{run}', exist_ok=True)
    shutil.copy(f, f'{dest}/{run}/ordinal_evaluation_results.json')
if os.path.exists('results/eo_summary.json'):
    shutil.copy('results/eo_summary.json', f'{dest}/eo_summary.json')
print('Saved eval results to', dest)